In [4]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV
)

from sklearn.preprocessing import StandardScaler

from imblearn.over_sampling import SMOTE

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

# ==========================================
# LOAD DATA
# ==========================================

df = pd.read_csv("../dataset/WA_Fn-UseC_-Telco-Customer-Churn.csv")

# ==========================================
# CLEANING
# ==========================================

df["TotalCharges"] = df["TotalCharges"].replace(" ", np.nan)

df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"]
)

df.dropna(inplace=True)

df.drop("customerID", axis=1, inplace=True)

# ==========================================
# FEATURE ENGINEERING
# ==========================================

df["AvgMonthlySpend"] = (
    df["TotalCharges"] /
    (df["tenure"] + 1)
)

df["LongTermCustomer"] = (
    df["tenure"] > 24
).astype(int)

df["HighMonthlyCharges"] = (
    df["MonthlyCharges"] > 70
).astype(int)

# ==========================================
# BINARY ENCODING
# ==========================================

binary_cols = [
    "gender",
    "Partner",
    "Dependents",
    "PhoneService",
    "PaperlessBilling",
    "Churn"
]

for col in binary_cols:

    df[col] = df[col].map({
        "Yes": 1,
        "No": 0,
        "Male": 1,
        "Female": 0
    })

# ==========================================
# ONE HOT ENCODING
# ==========================================

df = pd.get_dummies(
    df,
    drop_first=True
)

# ==========================================
# SPLIT
# ==========================================

X = df.drop("Churn", axis=1)

y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# ==========================================
# SCALING
# ==========================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

# ==========================================
# SMOTE
# ==========================================

smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train_scaled,
    y_train
)

print("\nBefore SMOTE:\n")
print(y_train.value_counts())

print("\nAfter SMOTE:\n")
print(y_train_smote.value_counts())

# ==========================================
# MODELS
# ==========================================

models = {

    "Logistic Regression":
        LogisticRegression(max_iter=2000),

    "Decision Tree":
        DecisionTreeClassifier(),

    "Random Forest":
        RandomForestClassifier(),

    "KNN":
        KNeighborsClassifier(),

    "SVM":
        SVC(probability=True),

    "Naive Bayes":
        GaussianNB(),

    "XGBoost":
        XGBClassifier(
            eval_metric='logloss'
        ),

    "LightGBM":
        LGBMClassifier(),

    "CatBoost":
        CatBoostClassifier(
            verbose=0
        )
}

# ==========================================
# TRAINING
# ==========================================

results = []

for name, model in models.items():

    model.fit(
        X_train_smote,
        y_train_smote
    )

    y_pred = model.predict(X_test_scaled)

    y_prob = model.predict_proba(
        X_test_scaled
    )[:,1]

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    roc_auc = roc_auc_score(
        y_test,
        y_prob
    )

    results.append([
        name,
        accuracy,
        roc_auc
    ])

    print("\n==============================")
    print(f"Model: {name}")
    print("==============================")

    print("Accuracy:", accuracy)

    print("ROC-AUC:", roc_auc)

    print("\nClassification Report:\n")

    print(classification_report(
        y_test,
        y_pred
    ))

# ==========================================
# FINAL RESULTS
# ==========================================

results_df = pd.DataFrame(
    results,
    columns=[
        "Model",
        "Accuracy",
        "ROC-AUC"
    ]
)

results_df = results_df.sort_values(
    by="ROC-AUC",
    ascending=False
)

print("\nFinal Results:\n")

print(results_df)

# ==========================================
# HYPERPARAMETER TUNING
# ==========================================

param_grid = {

    'C': [
        0.001,
        0.01,
        0.1,
        1,
        10,
        100
    ],

    'solver': [
        'liblinear',
        'saga'
    ],

    'penalty': [
        'l1',
        'l2'
    ],

    'class_weight': [
        None,
        'balanced'
    ]
}

grid = GridSearchCV(

    LogisticRegression(
        max_iter=3000
    ),

    param_grid,

    cv=5,

    scoring='roc_auc',

    n_jobs=-1
)

grid.fit(
    X_train_smote,
    y_train_smote
)

print("\nBest Parameters:")

print(grid.best_params_)

print("\nBest Cross Validation Score:")

print(grid.best_score_)

best_lr = grid.best_estimator_

y_pred = best_lr.predict(X_test_scaled)

y_prob = best_lr.predict_proba(
    X_test_scaled
)[:,1]

print("\nConfusion Matrix:\n")

print(confusion_matrix(
    y_test,
    y_pred
))

print("\nClassification Report:\n")

print(classification_report(
    y_test,
    y_pred
))

print(
    "Accuracy:",
    accuracy_score(
        y_test,
        y_pred
    )
)

print(
    "ROC-AUC:",
    roc_auc_score(
        y_test,
        y_prob
    )
)




Before SMOTE:

Churn
0    4130
1    1495
Name: count, dtype: int64

After SMOTE:

Churn
0    4130
1    4130
Name: count, dtype: int64

Model: Logistic Regression
Accuracy: 0.7377398720682303
ROC-AUC: 0.8358643378146824

Classification Report:

              precision    recall  f1-score   support

           0       0.90      0.72      0.80      1033
           1       0.50      0.78      0.61       374

    accuracy                           0.74      1407
   macro avg       0.70      0.75      0.71      1407
weighted avg       0.80      0.74      0.75      1407


Model: Decision Tree
Accuracy: 0.7121535181236673
ROC-AUC: 0.6589809029305641

Classification Report:

              precision    recall  f1-score   support

           0       0.82      0.77      0.80      1033
           1       0.46      0.55      0.50       374

    accuracy                           0.71      1407
   macro avg       0.64      0.66      0.65      1407
weighted avg       0.73      0.71      0.72      140